# Module 3: Identification Before Estimation

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

**Identification** is the question of whether the quantity you want is a
function of the distribution you can observe, given your assumptions. It is
answered before any data is touched, and no amount of estimation repairs a
failure of it.

This module writes the identifying assumptions for the designs in this
repository as checkable statements, and checks each one.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. The three questions, in order

| Step | Question | Answered by |
|---|---|---|
| **Estimand** | which quantity | Module 1 |
| **Identification** | is it a function of observables, under stated assumptions | Module 2 and this one |
| **Estimation** | how to compute it and how uncertain it is | everything else |

Most applied work starts at step three. The cost is that a failure at step
two produces a precise number with a small standard error and no meaning,
which is much harder to detect than a noisy one.

## 3. The assumptions, written out

For the difference in differences used throughout:

> **A1 Parallel trends.** E[Y(0) after − Y(0) before | D=1] = E[Y(0) after − Y(0) before | D=0]
>
> **A2 No anticipation.** Y(0) is the outcome for treated units in the pre period
>
> **A3 SUTVA.** One agency's treatment does not change another's outcome
>
> **A4 Stable composition.** The units and the measure mean the same thing throughout

Each has a diagnostic, and none of the diagnostics proves the assumption.

In [ ]:
pre = d[d["period"] == "before"].copy()
pre["tr"] = pre["agency_id"].isin(KEEP).astype(float)

# A1: pre period trend difference
z1 = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", pre,
             family=sm.families.Poisson(), offset=pre["lo"]).fit()
k1 = [x for x in z1.params.index if "yr" in x and "tr" in x][0]
l1, h1 = z1.conf_int().loc[k1]

# A2: a fake intervention one year before the real one
s = pre.copy()
s["fake"] = ((s["tr"] == 1) & (s["year_month"] >= "2022-07")).astype(float)
z2 = smf.glm("n_uof ~ C(agency_id) + C(year_month) + fake", s,
             family=sm.families.Poisson(), offset=s["lo"]).fit()
l2, h2 = z2.conf_int().loc["fake"]

# A3: a proxy, whether the comparison agencies moved unusually
s3 = d.copy()
s3["settled"] = ((s3["agency_id"].isin(KEEP)) & (s3["period"] == "after")).astype(float)
s3["phase"] = ((s3["agency_id"].isin(KEEP)) & (s3["period"] == "phase")).astype(float)
z3 = smf.glm("n_arrests ~ C(agency_id)+C(year_month)+settled+phase", s3,
             family=sm.families.Poisson()).fit()
l3, h3 = z3.conf_int().loc["settled"]

print("  assumption  diagnostic                          result")
print(f"  A1 parallel trends  pre period trend difference  "
      f"{pct(z1.params[k1]):+.2f}%/yr [{pct(l1):+.2f}, {pct(h1):+.2f}]")
print(f"  A2 no anticipation  fake intervention at 2022-07 "
      f"{pct(z2.params['fake']):+.2f}%   [{pct(l2):+.2f}, {pct(h2):+.2f}]")
print(f"  A3 SUTVA            effect on arrests            "
      f"{pct(z3.params['settled']):+.2f}%   [{pct(l3):+.2f}, {pct(h3):+.2f}]")
print(f"  A4 stable measure   see the data dictionary      "
      f"one agency reclassified calls in 2023-01, not the outcome used here")

Every diagnostic passes in the sense of not rejecting, and **not one of them
establishes its assumption.**

A1's interval reaches 3.31 percent a year, enough to matter. A2 tests
anticipation at one arbitrary date. A3 checks one of many possible spillover
routes. A4 rests on reading the documentation.

**That is the normal state of an observational design**, and the write up
should say so in those terms rather than reporting that the assumptions were
"verified".

## 4. What identification failure looks like

An unidentified design does not announce itself. It produces an estimate with
a standard error.

In [ ]:
rows = []
for lab, form in [
        ("agency and month effects, A007 out",
         "n_uof ~ C(agency_id)+C(year_month)+settled+phase"),
        ("the same, no time term at all",
         "n_uof ~ C(agency_id)+settled+phase"),
        ("the same, no agency term at all",
         "n_uof ~ C(year_month)+settled+phase")]:
    e, lo, hi, z = fit(d, KEEP, form=form)
    rows.append({"specification": lab, "estimate": f"{e:+.1f}%",
                 "interval width": round(hi - lo, 1),
                 "AIC": round(z.aic, 0)})
print(f"  the truth is {TRUTH:+.1f}%\n")
pd.DataFrame(rows).set_index("specification")

The specification with no time term has a **narrower** interval than the
identified one, 6.8 points against 11.0, and it is wrong by 17 percentage
points. Precision is not evidence of identification.

AIC does happen to prefer the identified specification here. **That is luck
rather than a property.** AIC compares fit, and identification is not about
fit: a model can fit better while estimating something other than the effect,
which is exactly what agency specific trends did in Intermediate Module 8.

**This is the argument for step two.** The only thing that distinguishes the
first row from the others is an argument made before the models were run.

## 5. Identification for the designs this series does not use

| Design | What it needs | Available here |
|---|---|---|
| Randomisation | assignment independent of potential outcomes | no, selection was on the outcome |
| Matching | overlap in the confounders | no, see Module 10 |
| Instrumental variables | relevance, exclusion, monotonicity | no instrument exists, Module 11 |
| Regression discontinuity | a sharp threshold with continuity around it | the rule is close, not sharp, Module 12 |
| Difference in differences | parallel trends | yes, with the caveat on A1 |

**Four of five are unavailable, and each is unavailable for a reason that can
be stated in one line.** Part III works through them properly.

## Exercise

A2, no anticipation, was tested at one date. Test it at every date in the last
two pre program years and see whether the conclusion depends on the choice.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for cut in ["2021-07", "2022-01", "2022-07", "2023-01"]:
        s = pre.copy()
        s["fake"] = ((s["tr"] == 1) & (s["year_month"] >= cut)).astype(float)
        z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + fake", s,
                    family=sm.families.Poisson(), offset=s["lo"]).fit()
        lo, hi = z.conf_int().loc["fake"]
        rows.append({"anticipation tested from": cut,
                     "estimate": f"{pct(z.params['fake']):+.1f}%",
                     "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]",
                     "covers zero": "yes" if lo < 0 < hi else "NO"})
    display(pd.DataFrame(rows).set_index("anticipation tested from"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Every date's interval covers zero, so the conclusion does not depend on which
was chosen. The point estimates drift more negative as the date moves later,
which is the residual pre trend difference accumulating and is the same drift
Intermediate Module 12 found.

**The value of running all four is that the choice becomes visible.** A single
test at a date picked after seeing results is a specification search with one
degree of freedom, and it is indistinguishable in the output from a test
chosen in advance.

Note also what this test can and cannot detect. It finds anticipation that
takes the form of a level shift at a particular month. An agency that began
changing gradually two years out, in proportion to how likely it was to be
selected, would produce no level shift anywhere and pass every date.

</details>

---

**Next:** [Module 4: Selection Mechanisms and What They Do to an Estimate](Module_04_Selection_Mechanisms.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*